In [1]:
import pandas as pd
import json
import mne
import numpy as np
import mne
import os
import keras 
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [2]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [3]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [4]:
nan_counts = df.isna().sum()
print(nan_counts)

participant_id       0
subject_id           0
group                0
updrs_part_iii       5
updrs_total          5
moca                 4
age                  0
sex                  0
disease_duration    28
ledd                29
pigd_score          31
td_score            31
ctt                  9
dtype: int64


In [5]:
missing_ids = []

for i in range(1, 145):
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

    # Check if file exists; if not, store or print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

In [6]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"

    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [7]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_58/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [8]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [9]:

def get_eeg_signal(sub_id, task="walk", band="full",target_sfreq=256,duration=2.0, notch_freq=50.0,
    reject_threshold=0.00028,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, and segments EEG data.
    Returns float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }

    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")

    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & isolate EEG
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter (only if applicable to selected band range)
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency (e.g., 128 Hz)
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. Epoching with Artifact Rejection
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw,
        events=events,
        tmin=0,
        tmax=duration - (1 / target_sfreq),  # Exactly 256 samples at 128 Hz
        baseline=None,
        reject=reject_criteria,
        preload=True,
        verbose=False
    )

    # Extract array and cast to float32 to save RAM
    signal = epochs.get_data().astype(np.float32)

    return signal


# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


Beta Walk Signal shape (N, C, T): (121, 60, 512)
Full Rest Signal shape (N, C, T): (121, 60, 512)


In [10]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [11]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [12]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []

    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids

    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)

            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)

        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")

    return X_hc, X_pd


In [13]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]

    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)

    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active

        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break

        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0

                available = maj_counts[i] - allocations[i]
                take = min(share, available)

                allocations[i] += take
                remaining_target -= take

                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])

    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [14]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, Model, models

class SeriesDecomp(layers.Layer):
    """Extracts seasonal and trend components using moving average pooling."""
    def __init__(self, kernel_size=25, **kwargs):
        super(SeriesDecomp, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        self.avg = layers.AveragePooling1D(
            pool_size=kernel_size, 
            strides=1, 
            padding='same'
        )

    def call(self, x):
        trend = self.avg(x)
        seasonal = x - trend
        return seasonal, trend


class FourierBlock(layers.Layer):
    """Frequency Enhanced Block (FEB) using rFFT in O(L) complexity."""
    def __init__(self, in_channels, out_channels, modes=16, **kwargs):
        super(FourierBlock, self).__init__(**kwargs)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        self.scale = 1.0 / (in_channels * out_channels)

    def build(self, input_shape):
        self.weights_real = self.add_weight(
            name="weights_real",
            shape=(self.in_channels, self.out_channels, self.modes),
            initializer=tf.keras.initializers.RandomNormal(stddev=self.scale),
            trainable=True
        )
        self.weights_imag = self.add_weight(
            name="weights_imag",
            shape=(self.in_channels, self.out_channels, self.modes),
            initializer=tf.keras.initializers.RandomNormal(stddev=self.scale),
            trainable=True
        )
        super(FourierBlock, self).build(input_shape)

    def call(self, x):
        # Dynamically extract sequence length directly from the input tensor shape
        target_seq_len = tf.shape(x)[1]
        
        # x shape: [batch, seq_len, in_channels]
        x_trans = tf.transpose(x, perm=[0, 2, 1])  # [batch, in_channels, seq_len]
        x_ft = tf.signal.rfft(x_trans)            # [batch, in_channels, rfft_len]
        
        rfft_len = tf.shape(x_ft)[-1]
        modes_to_keep = tf.minimum(self.modes, rfft_len)
        
        x_ft_sampled = x_ft[:, :, :modes_to_keep]
        
        weights = tf.complex(
            self.weights_real[:, :, :modes_to_keep], 
            self.weights_imag[:, :, :modes_to_keep]
        )
        out_ft = tf.einsum("bim,iom->bom", x_ft_sampled, weights)
        
        pad_size = rfft_len - modes_to_keep
        out_ft_padded = tf.pad(out_ft, [[0, 0], [0, 0], [0, pad_size]])
        
        res = tf.signal.irfft(out_ft_padded, fft_length=[target_seq_len])
        return tf.transpose(res, perm=[0, 2, 1])


class MultiHeadFourierAttention(layers.Layer):
    """Multi-Head Frequency Enhanced Attention."""
    def __init__(self, d_model, n_heads=4, modes=16, seq_len=None, **kwargs):
        super(MultiHeadFourierAttention, self).__init__(**kwargs)
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.modes = modes
        
        self.value_proj = layers.Dense(d_model)
        self.out_proj = layers.Dense(d_model)
        
        self.fourier_blocks = [
            FourierBlock(self.head_dim, self.head_dim, modes=modes)
            for _ in range(n_heads)
        ]

    def build(self, input_shape):
        # Build projections explicitly to prevent shape inference errors
        self.value_proj.build(input_shape)
        self.out_proj.build((input_shape[0], input_shape[1], self.d_model))
        super(MultiHeadFourierAttention, self).build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]

        v = self.value_proj(x)
        v_heads = tf.reshape(v, (batch_size, seq_len, self.n_heads, self.head_dim))
        v_heads = tf.transpose(v_heads, perm=[0, 2, 1, 3])  # [batch, n_heads, seq_len, head_dim]

        head_outputs = []
        for h in range(self.n_heads):
            out_h = self.fourier_blocks[h](v_heads[:, h, :, :])
            head_outputs.append(out_h)

        context = tf.concat(head_outputs, axis=-1)
        # Force static last dimension shape for Keras Layer internal tracking
        context.set_shape([None, None, self.d_model])
        return self.out_proj(context)


class FEDformerEncoderLayer(layers.Layer):
    """FEDformer Encoder Layer integrating Frequency Attention & Series Decomposition."""
    def __init__(self, d_model, d_ff, n_heads=4, modes=16, seq_len=None, moving_avg=25, dropout=0.1, **kwargs):
        super(FEDformerEncoderLayer, self).__init__(**kwargs)
        self.d_model = d_model
        self.decomp1 = SeriesDecomp(kernel_size=moving_avg)
        self.freq_attn = MultiHeadFourierAttention(d_model=d_model, n_heads=n_heads, modes=modes)
        self.decomp2 = SeriesDecomp(kernel_size=moving_avg)
        
        self.ffn = models.Sequential([
            layers.Dense(d_ff, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(d_model),
            layers.Dropout(dropout)
        ])
        self.norm = layers.LayerNormalization(epsilon=1e-5)

    def build(self, input_shape):
        self.freq_attn.build(input_shape)
        self.ffn.build(input_shape)
        self.norm.build(input_shape)
        super(FEDformerEncoderLayer, self).build(input_shape)

    def call(self, x):
        seasonal_x, _ = self.decomp1(x)
        attn_out = self.freq_attn(seasonal_x)
        x = seasonal_x + attn_out

        seasonal_x, _ = self.decomp2(x)
        ffn_out = self.ffn(seasonal_x)
        x = seasonal_x + ffn_out

        return self.norm(x)


class FEDformerClassifier(Model):
    """FEDformer Classifier with Metric Learning Prototypes."""
    def __init__(self, channels=60, seq_len=None, d_model=64, d_ff=128, latent_dim=16, 
                 num_layers=2, n_heads=4, modes=16, moving_avg=25, **kwargs):
        super(FEDformerClassifier, self).__init__(**kwargs)
        self.channels = channels
        self.seq_len = seq_len
        
        self.enc_embedding = layers.Dense(d_model)
        self.encoder_layers = [
            FEDformerEncoderLayer(
                d_model=d_model, d_ff=d_ff, n_heads=n_heads, 
                modes=modes, moving_avg=moving_avg
            ) 
            for _ in range(num_layers)
        ]
        
        self.global_pool = layers.GlobalAveragePooling1D()
        self.fc_reduce = layers.Dense(latent_dim, activation='relu')
        self.fc_mu = layers.Dense(latent_dim)
        
        self.pd_prototype = self.add_weight(
            name="pd_prototype",
            shape=(1, latent_dim),
            initializer="random_normal",
            trainable=True
        )
        self.hc_prototype = self.add_weight(
            name="hc_prototype",
            shape=(1, latent_dim),
            initializer="random_normal",
            trainable=True
        )
        
        self.logit_scale = self.add_weight(
            name="logit_scale",
            shape=(),
            initializer=tf.keras.initializers.Constant(10.0),
            trainable=True
        )

    def call(self, inputs, training=False):
        # Input shape expected: [batch, channels, seq_len] -> Transpose to [batch, seq_len, channels]
        x = tf.transpose(inputs, perm=[0, 2, 1])
        x = self.enc_embedding(x)
        
        for layer in self.encoder_layers:
            x = layer(x)
            
        x = self.global_pool(x)
        x = self.fc_reduce(x)
        z = self.fc_mu(x)
        
        z_norm = tf.math.l2_normalize(z, axis=-1)
        pd_norm = tf.math.l2_normalize(self.pd_prototype, axis=-1)
        hc_norm = tf.math.l2_normalize(self.hc_prototype, axis=-1)
        
        sim_hc = tf.reduce_sum(z_norm * hc_norm, axis=-1, keepdims=True)
        sim_pd = tf.reduce_sum(z_norm * pd_norm, axis=-1, keepdims=True)
        
        logits = tf.concat([sim_hc, sim_pd], axis=-1) * self.logit_scale
        probs = tf.nn.softmax(logits, axis=-1)
        
        return probs[:, 1:2]

In [16]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import product

def run_subject_level_fedformer_cv_optimized(X_healthy, X_pd, SEED=42, epochs=50):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)
    
    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))
    
    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))
    
    total_correct = 0
    total_subjects = 0
    fold_summary_records = []
    
    # Define Hyperparameter Lists for FEDformer Grid Search
    param_grid = {
        'lr': [1e-4],
        'batch_size': [32],
        'd_model': [64],
        'd_ff': [128],
        'num_layers': [2],
        'n_heads': [4],
        'modes': [16],        # Key FEDformer hyperparameter: number of frequency modes
        'moving_avg': [25],
        'seq_len': [X_healthy[0].shape[1]] # Automatically matches input time dimension (T)
    }
    
    # Generate all possible hyperparameter combinations
    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]
    
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 (FEDFORMER) ==========")
        print(f"========================================")
        
        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]
        
        # --- INNER LOOP: Grid Search Hyperparameter Optimization ---
        best_score = -1.0
        best_params = None
        best_threshold = 75
        
        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))
        
        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []
            
            for inner_fold in range(3):
                tf.keras.backend.clear_session()
                
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]
                
                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]
                
                # 1. Balance classes for inner training (Subject-Level)
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_tr = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_tr = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)
                
                # Shuffle training epochs safely with dynamic seed
                rng = np.random.RandomState(SEED + inner_fold)
                shuffle_idx = rng.permutation(len(X_tr))
                X_tr = X_tr[shuffle_idx]
                y_tr = y_tr[shuffle_idx]
                
                # Dynamically infer number of channels from data shape (N, C, T)
                num_channels = X_tr.shape[1]
                
                inner_model = FEDformerClassifier(
                    channels=num_channels,
                    seq_len=params['seq_len'],
                    d_model=params['d_model'],
                    d_ff=params['d_ff'],
                    latent_dim=16,
                    num_layers=params['num_layers'],
                    n_heads=params['n_heads'],
                    modes=params['modes'],
                    moving_avg=params['moving_avg']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']), 
                    loss='binary_crossentropy'
                )
                
                # Fit Inner Model (Fixed Epochs, No Early Stopping / Validation Split)
                inner_model.fit(
                    X_tr, y_tr, 
                    epochs=epochs, 
                    batch_size=params['batch_size'], 
                    verbose=0
                )
                
                # Tune decision threshold on inner validation set
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)
                
                val_subject_ratios = []
                valid_val_labels = []
                
                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = np.asarray(sub, dtype=np.float32)
                    if sub_array.shape[0] == 0:
                        continue
                
                    epoch_probs = inner_model.predict(sub_array, verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)
                
                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t
                
                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)
            
            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))
        
        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")
        
        # --- OUTER TRAINING & FINAL FIT ---
        tf.keras.backend.clear_session()
        
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]
        
        X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_tr_f = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
        y_tr_f = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)
        
        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_tr_f))
        X_tr_f, y_tr_f = X_tr_f[shuffle_idx_final], y_tr_f[shuffle_idx_final]
        
        num_channels = X_tr_f.shape[1]
        
        final_model = FEDformerClassifier(
            channels=num_channels,
            seq_len=best_params['seq_len'],
            d_model=best_params['d_model'],
            d_ff=best_params['d_ff'],
            latent_dim=16,
            num_layers=best_params['num_layers'],
            n_heads=best_params['n_heads'],
            modes=best_params['modes'],
            moving_avg=best_params['moving_avg']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']), 
            loss='binary_crossentropy'
        )
        
        # Track Training Time (Fixed Epochs, No Early Stopping)
        start_time = time.time()
        final_model.fit(
            X_tr_f, y_tr_f, 
            epochs=epochs, 
            batch_size=best_params['batch_size'], 
            verbose=0
        )
        training_time = time.time() - start_time
        
        # Track Model Parameters Count (Placed safely AFTER fit)
        model_params_count = final_model.count_params()
        
        # Test evaluation on the held-out outer fold subjects using Majority Voting
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)
        
        hc_correct_count = 0
        pd_correct_count = 0
        
        min_t, max_t = thresholds[0], thresholds[-1]
        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = np.asarray(sub, dtype=np.float32)
            if sub_array.shape[0] == 0:
                continue
            
            pct_pd = float(np.mean(final_model.predict(sub_array, verbose=0).flatten()) * 100)
            
            vote_thresholds = [
                max(min_t, best_threshold - 5),
                best_threshold,
                min(max_t, best_threshold + 5)
            ]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0
            
            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1
                
        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)
        
        total_correct += fold_total_correct
        total_subjects += fold_total_subjects
        
        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0
        
        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}",
            'Training Time (s)': f"{training_time:.2f}",
            'Model Parameters': f"{model_params_count:,}"
        })
        
        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}% | Time: {training_time:.2f}s | Params: {model_params_count:,}")

    # Summary Generation
    summary_df = pd.DataFrame(fold_summary_records)
    total_hc_correct = sum(int(x.split('/')[0]) for x in summary_df['Healthy Correct'])
    total_hc_subjects = sum(int(x.split('/')[1]) for x in summary_df['Healthy Correct'])
    
    total_pd_correct = sum(int(x.split('/')[0]) for x in summary_df['PD Correct'])
    total_pd_subjects = sum(int(x.split('/')[1]) for x in summary_df['PD Correct'])
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0
    
    print(f"\n========================================")
    print(f"Healthy Controls Correct: {total_hc_correct}/{total_hc_subjects}")
    print(f"Parkinson's Disease (PD) Correct: {total_pd_correct}/{total_pd_subjects}")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy (FEDformer): {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))
    
    return summary_df

In [17]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_fedformer_cv_optimized(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 2/5 | PD: 19/23 | Acc: 75.00% | Time: 300.61s | Params: 120,833

========== OUTER FOLD 5 / 5 (FEDFORMER) ==========


2026-09-09 08:39:43.176928: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-09 08:43:03.038471: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60} | Threshold: 65% (Inner Acc: 0.8192)


2026-09-09 08:50:33.069354: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-09 08:55:30.029242: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 2/5 | PD: 19/23 | Acc: 75.00% | Time: 299.08s | Params: 120,833

Healthy Controls Correct: 16/28
Parkinson's Disease (PD) Correct: 96/116
Total Combined Correct: 112/144
Overall Nested Cross-Validation Accuracy (FEDformer): 77.78%

--- Nested Cross-Validation Summary ---
 Fold Number                                                                                                                       Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Model Parameters
           1 {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60}                     70             4/6      20/24            80.00%         24/30            288.00          120,833
           2 {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60}           

In [18]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_fedformer_cv_optimized(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 2/4 | PD: 19/22 | Acc: 80.77% | Time: 215.52s | Params: 120,833

========== OUTER FOLD 5 / 5 (FEDFORMER) ==========


2026-09-09 09:44:52.010295: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-09 09:47:19.549629: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60} | Threshold: 65% (Inner Acc: 0.7966)


2026-09-09 09:53:12.752183: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-09 09:57:00.821218: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 2/4 | PD: 22/22 | Acc: 92.31% | Time: 229.69s | Params: 120,833

Healthy Controls Correct: 11/23
Parkinson's Disease (PD) Correct: 95/110
Total Combined Correct: 106/133
Overall Nested Cross-Validation Accuracy (FEDformer): 79.70%

--- Nested Cross-Validation Summary ---
 Fold Number                                                                                                                       Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Model Parameters
           1 {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60}                     65             4/5      19/22            85.19%         23/27            227.51          120,833
           2 {'lr': 0.0001, 'batch_size': 32, 'd_model': 64, 'd_ff': 128, 'num_layers': 2, 'n_heads': 4, 'modes': 16, 'moving_avg': 25, 'seq_len': 60}           

In [ ]:
healthy = 11/23 

PD = 95/110